# MSCS 634 – Project Deliverable 1
## Data Collection, Cleaning, and Exploration

**Dataset:** UCI Online Retail  
**Research focus:** Understanding historical online retail transactions, product demand, customer purchasing behavior, and sales patterns.

### Project objective
This notebook loads and inspects the UCI Online Retail dataset, documents data-quality issues, prepares an analysis-ready sales dataset, and uses exploratory data analysis (EDA) to identify patterns that can guide later regression, classification, clustering, and association-rule mining.

**Important:** The source data are historical transactions from December 2010 through December 2011. Findings describe this dataset and should not be interpreted as current retail-market conditions.

## 1. Dataset description and justification

The UCI Online Retail dataset contains 541,909 transaction-line records from a UK-based, non-store online retailer. The source covers transactions from 1 December 2010 through 9 December 2011. It contains eight columns:

- `InvoiceNo`: invoice/transaction identifier; values beginning with `C` indicate cancellations.
- `StockCode`: product identifier.
- `Description`: product description.
- `Quantity`: units recorded on the transaction line; negative values can represent returns.
- `InvoiceDate`: transaction date and time.
- `UnitPrice`: unit price in pounds sterling.
- `CustomerID`: customer identifier; some records have no customer identifier.
- `Country`: customer country.

This dataset is appropriate because it exceeds the required 500-record threshold, has the required minimum of eight attributes, and supports several complementary data-mining tasks. Transaction-level data can support revenue prediction and cancellation classification; customer-level summaries can support segmentation; and invoice-product baskets can support association-rule mining.

**Source:** Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33. Licensed under CC BY 4.0.

In [ ]:
# Install any missing packages in the active notebook environment.
# Uncomment the next line if these packages are not already installed.
# %pip install pandas numpy matplotlib seaborn openpyxl scikit-learn

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

RANDOM_STATE = 42
DATA_DIR = "data"
FIGURE_DIR = "visualizations"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

## 2. Load the dataset

The original Excel file is downloaded from the UCI repository. If the download is blocked in your environment, download `Online Retail.xlsx` manually from the [UCI dataset page](https://archive.ics.uci.edu/dataset/352/online+retail), place it in the `data` folder, and rerun the cell.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

file_path = Path(DATA_DIR) / "Online Retail.xlsx"
source_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

if not file_path.exists():
    try:
        print("Downloading the UCI Online Retail dataset...")
        urlretrieve(source_url, file_path)
    except Exception as exc:
        raise RuntimeError(
            "Automatic download failed. Download Online Retail.xlsx from "
            "https://archive.ics.uci.edu/dataset/352/online+retail and save it "
            "inside the data/ folder, then rerun this cell."
        ) from exc

raw_df = pd.read_excel(file_path, engine="openpyxl")
print(f"Dataset shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")
display(raw_df.head())

## 3. Inspect the dataset structure

The following checks review data types, summary statistics, unique values, and missing-value counts before any rows are removed or transformed.

In [ ]:
print("Data types and non-null counts:")
raw_df.info()

print("\\nDescriptive statistics (numeric columns):")
display(raw_df.describe().T)

print("\\nUnique values per column:")
display(raw_df.nunique(dropna=True).rename("unique_values").to_frame())

print("\\nMissing values:")
missing_summary = pd.DataFrame({
    "missing_count": raw_df.isna().sum(),
    "missing_percent": (raw_df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)
display(missing_summary)

## 4. Data cleaning and preparation

Cleaning decisions are documented rather than applied blindly:

1. Standardize column names and trim text fields.
2. Parse `InvoiceDate` as a datetime value and coerce invalid dates to missing.
3. Remove exact duplicate transaction lines.
4. Preserve cancellation/return records in a separate cleaned transaction table. Negative quantities and invoices beginning with `C` can carry meaningful operational information and are not automatically treated as ordinary sales.
5. For sales-focused EDA, create a separate subset with positive quantity, positive unit price, and non-cancelled invoices. This prevents returns/cancellations and non-positive prices from distorting sales summaries.
6. Do not discard all rows with missing `CustomerID`: these rows can still contribute to product and country-level analysis. Customer-level modeling will require known customer identifiers.
7. Flag potential outliers using the IQR rule. Do not automatically delete high-value or high-quantity records because wholesale purchases may be legitimate.

These choices retain useful business information while creating a clearly defined subset for sales analysis.

In [ ]:
# Work on a copy so the original imported data remain unchanged.
df = raw_df.copy()

# Standardize column labels and clean text fields.
df.columns = df.columns.str.strip()
for col in ["InvoiceNo", "StockCode", "Description", "Country"]:
    df[col] = df[col].astype("string").str.strip()

# Normalize product descriptions for consistent grouping.
df["Description"] = df["Description"].str.upper()

# Convert dates and numeric fields to appropriate data types.
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

# Record data-quality counts before cleaning.
quality_before = {
    "rows_before": len(df),
    "exact_duplicates": int(df.duplicated().sum()),
    "invalid_dates": int(df["InvoiceDate"].isna().sum()),
    "missing_descriptions": int(df["Description"].isna().sum()),
    "missing_customer_ids": int(df["CustomerID"].isna().sum()),
    "nonpositive_quantities": int((df["Quantity"] <= 0).sum()),
    "nonpositive_unit_prices": int((df["UnitPrice"] <= 0).sum()),
}
display(pd.Series(quality_before, name="count").to_frame())

# Remove exact duplicate rows and rows without usable date/quantity/price values.
df = df.drop_duplicates().copy()
df = df.dropna(subset=["InvoiceDate", "Quantity", "UnitPrice"]).copy()

# Calculate line revenue. Negative quantities remain negative and can represent returns.
df["LineRevenue"] = df["Quantity"] * df["UnitPrice"]

# A cancellation marker is explicitly identified from the invoice number.
df["IsCancellation"] = df["InvoiceNo"].str.upper().str.startswith("C", na=False)

# Build a sales-only subset for sales-focused EDA and later sales modeling.
sales_df = df[
    (~df["IsCancellation"]) &
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()

# Keep a separate customer-identified subset for customer-level analysis.
customer_sales_df = sales_df.dropna(subset=["CustomerID"]).copy()

print(f"Rows after general cleaning: {len(df):,}")
print(f"Sales-only rows: {len(sales_df):,}")
print(f"Sales rows with identified customers: {len(customer_sales_df):,}")
print(f"Exact duplicates removed: {quality_before['exact_duplicates']:,}")
display(sales_df.head())

### 4.1 Review remaining missing values and data-quality decisions

Missing customer identifiers are retained in the sales table because removing them would discard otherwise usable transactions. Missing product descriptions are also not imputed with invented product names; rows without descriptions will be excluded only from product-description-based analyses. The cell below summarizes the remaining missingness.

In [ ]:
remaining_missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
})
display(remaining_missing.sort_values("missing_percent", ascending=False))

print("Cancellation invoice rows:", int(df["IsCancellation"].sum()))
print("Rows with negative quantity:", int((df["Quantity"] < 0).sum()))
print("Rows with zero or negative unit price:", int((df["UnitPrice"] <= 0).sum()))
print("Rows with missing CustomerID:", int(df["CustomerID"].isna().sum()))

## 5. Exploratory data analysis (EDA)

The EDA focuses on distributions, outliers, time patterns, country-level activity, and product demand. Visualizations are saved to the `visualizations/` directory so they can be included in the GitHub repository and final report.

In [ ]:
# 5.1 Distribution of quantity and unit price.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(sales_df["Quantity"].clip(upper=sales_df["Quantity"].quantile(0.99)),
             bins=50, ax=axes[0], kde=True)
axes[0].set_title("Distribution of Quantity (capped at 99th percentile)")
axes[0].set_xlabel("Quantity")
sns.histplot(sales_df["UnitPrice"].clip(upper=sales_df["UnitPrice"].quantile(0.99)),
             bins=50, ax=axes[1], kde=True)
axes[1].set_title("Distribution of Unit Price (capped at 99th percentile)")
axes[1].set_xlabel("Unit price (£)")
plt.tight_layout()
plt.savefig(f"{FIGURE_DIR}/quantity_unitprice_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 5.2 Boxplots to inspect extreme values.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x=sales_df["Quantity"].clip(upper=sales_df["Quantity"].quantile(0.99)), ax=axes[0])
axes[0].set_title("Quantity Outlier Inspection (99th percentile cap)")
axes[0].set_xlabel("Quantity")
sns.boxplot(x=sales_df["UnitPrice"].clip(upper=sales_df["UnitPrice"].quantile(0.99)), ax=axes[1])
axes[1].set_title("Unit Price Outlier Inspection (99th percentile cap)")
axes[1].set_xlabel("Unit price (£)")
plt.tight_layout()
plt.savefig(f"{FIGURE_DIR}/quantity_unitprice_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

# IQR-based flags are diagnostic only; flagged records are not deleted automatically.
def iqr_outlier_count(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((series < lower) | (series > upper)).sum()), lower, upper

for col in ["Quantity", "UnitPrice", "LineRevenue"]:
    count, lower, upper = iqr_outlier_count(sales_df[col])
    print(f"{col}: {count:,} IQR-flagged rows; lower={lower:,.2f}, upper={upper:,.2f}")

In [ ]:
# 5.3 Monthly revenue trend.
monthly_revenue = (
    sales_df.set_index("InvoiceDate")
    .resample("MS")["LineRevenue"]
    .sum()
)
plt.figure(figsize=(12, 5))
monthly_revenue.plot(marker="o")
plt.title("Monthly Revenue in the Historical Dataset")
plt.xlabel("Month")
plt.ylabel("Revenue (£)")
plt.tight_layout()
plt.savefig(f"{FIGURE_DIR}/monthly_revenue.png", dpi=150, bbox_inches="tight")
plt.show()
display(monthly_revenue.to_frame("revenue_gbp"))

In [ ]:
# 5.4 Top countries by sales revenue.
country_revenue = (sales_df.groupby("Country")["LineRevenue"]
                   .sum().sort_values(ascending=False).head(10))
plt.figure(figsize=(11, 6))
sns.barplot(x=country_revenue.values, y=country_revenue.index)
plt.title("Top 10 Countries by Recorded Sales Revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("Country")
plt.tight_layout()
plt.savefig(f"{FIGURE_DIR}/top_countries_revenue.png", dpi=150, bbox_inches="tight")
plt.show()
display(country_revenue.to_frame("revenue_gbp"))

In [ ]:
# 5.5 Top products by units sold (exclude missing descriptions).
top_products = (sales_df.dropna(subset=["Description"])
                .groupby("Description")["Quantity"]
                .sum().sort_values(ascending=False).head(10))
plt.figure(figsize=(12, 6))
sns.barplot(x=top_products.values, y=top_products.index)
plt.title("Top 10 Products by Units Sold")
plt.xlabel("Units sold")
plt.ylabel("Product")
plt.tight_layout()
plt.savefig(f"{FIGURE_DIR}/top_products_quantity.png", dpi=150, bbox_inches="tight")
plt.show()
display(top_products.to_frame("units_sold"))

In [ ]:
# 5.6 Correlation among numeric sales variables.
numeric_corr = sales_df[["Quantity", "UnitPrice", "LineRevenue"]].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(numeric_corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Matrix for Sales Variables")
plt.tight_layout()
plt.savefig(f"{FIGURE_DIR}/numeric_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
display(numeric_corr)

## 6. Summary statistics and preliminary insights

Run the next cell to generate dataset-specific figures from the downloaded file. Do not copy expected or illustrative values into the report before running the notebook; use the printed results from your own execution.

In [ ]:
summary = {
    "general_cleaned_rows": len(df),
    "sales_only_rows": len(sales_df),
    "unique_invoices_in_sales": sales_df["InvoiceNo"].nunique(),
    "unique_products_in_sales": sales_df["StockCode"].nunique(),
    "identified_customers": customer_sales_df["CustomerID"].nunique(),
    "countries_in_sales": sales_df["Country"].nunique(),
    "total_sales_revenue_gbp": sales_df["LineRevenue"].sum(),
    "average_line_revenue_gbp": sales_df["LineRevenue"].mean(),
    "date_start": sales_df["InvoiceDate"].min(),
    "date_end": sales_df["InvoiceDate"].max(),
}
display(pd.Series(summary, name="value").to_frame())

print("\\nInterpretation prompts:")
print("1. Which months show the highest and lowest recorded revenue?")
print("2. Is revenue concentrated in the United Kingdom or distributed across countries?")
print("3. Are quantity and unit-price distributions skewed?")
print("4. Which products appear most frequently by total units sold?")
print("5. How might missing CustomerID values affect customer segmentation?")
print("6. What business explanations might account for extreme quantities or prices?")

## 7. How EDA guides future modeling

- **Regression (Deliverable 2):** Use transaction-level features such as quantity, unit price, time components, and product/country encodings to estimate line revenue. Avoid data leakage by defining the prediction target and available predictors carefully. Because `LineRevenue = Quantity × UnitPrice`, a model using both components to predict line revenue is mathematically straightforward; a more meaningful extension is to predict invoice-level or future-period revenue using information available before the outcome.
- **Classification (Deliverable 3):** Cancellation status can be used as a target, but class balance must be checked and evaluation should include precision, recall, F1, and a confusion matrix. Features that directly reveal cancellation status must not be used as predictors.
- **Clustering:** Build customer-level RFM features (recency, frequency, monetary value) using records with known `CustomerID`. Standardize skewed variables and compare cluster quality.
- **Association rules:** Construct invoice-product baskets from valid positive-quantity sales. Exclude missing product descriptions and choose support/confidence thresholds carefully to avoid overwhelming numbers of trivial rules.
- **Outliers:** Retain plausible wholesale orders unless domain review identifies errors. Consider robust scaling, log transforms, or sensitivity analysis rather than arbitrary deletion.

## 8. Limitations and ethical considerations

The dataset is historical and comes from one retailer; results may not generalize to other businesses or current market conditions. Some transactions lack customer identifiers, limiting customer-level analysis. Product descriptions and transaction codes may contain inconsistencies. Customer identifiers should be treated as potentially linkable data and should not be published in unnecessary detail. Any predictive model should be checked for bias, leakage, and unintended consequences before being used to make decisions about customers.

## 9. Conclusion

This deliverable establishes a reproducible workflow for importing, inspecting, cleaning, and exploring the UCI Online Retail dataset. It distinguishes general transaction cleaning from the sales-only analytical subset, preserves returns and cancellations for separate analysis, documents missing customer identifiers, and flags rather than automatically deletes potential outliers. The EDA outputs provide a foundation for the regression, classification, clustering, and association-rule tasks in later deliverables.

## References

Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33

Chen, D., Sain, S. L., & Guo, K. (2012). Data mining for the online retail industry: A case study of RFM model-based customer segmentation using data mining. *Journal of Database Marketing & Customer Strategy Management, 19*, 197–208. https://doi.org/10.1057/dbm.2012.17
